In [1]:
import numpy as np
from pathlib import Path
import timeit
import pandas as pd

In [19]:
def update_equation(D,A=None,pc=None,pr=None,order='ST'):
    """ 
    update_equation updates the probability(ies) to be degraded, D, by one time step, 
    given the interaction matrix A, collapse rate pc and recovery rate pr

    Parameters
    ---
    D : numpy.array
        The initial array of degraded probabilities
    A : numpy.array
        The interaction matrix (i.e. adjacency matrix), must be of shape (len(D), len(D)). 
        No self loops, i.e. all diagonal elements should be 0.
        Default is np.zeros([len(D)]*2) i.e. the empty graph, such that all Markov chains are disconnected.
    pc : numpy.array or something broadcastable
        The array/list/collection of collapse rates. Must be in [0,1]. 
        If none supplied, default is 1/2 for all Markov chains
    pr : numpy.array or something broadcastable
        The array/list/collection of collapse rates. Must be in [0,1]
        If none supplied default is tr = 10tc where tr is the recovery time and 
        tc the collapse time with pc=1/(tc+1)
    order : str
        The order of the interaction matrix, 
        if 'ST' then the interaction matrix is indexed as A[source, target]
        if 'TS' then the interaction matrix is indexed as A[target, source]
        Default is 'ST'
        
    Returns
    ---
    D_out : numpy.array

    Examples
    ---
    For an initial vector of zeros, regardless of the interaction matrix, D_out will always be pc
    >>> update_equation(np.zeros(2), A = np.array( [[0, -2], [4, 0]] ), pc = np.array([0.2, 0.9]), order='ST')
    array([0.2, 0.9]) 
    
    For an initial vector of ones, regardless of the interaction matrix, D_out will always be pr
    >>> update_equation(np.ones(2), A = np.array( [[0, -2], [4, 0]] ), pc = np.array([0.2, 0.9]), pr = np.array([0.5, 0.5]), order='ST')
    array([0.5, 0.5])

    For non-trivial initial D:
    >>> update_equation(np.array([0, 1]), A = np.array( [[0, -2], [4, 0]] ), pc = np.array([0.2, 0.9]), 
        >>> pr = np.array([0.5, 0.5]), order='ST')
    array([1, 0.5])
    >>> update_equation_quick(np.array([0.5, 0.5]), np.array( [[0, 1], [1, 0]] ), np.array([0.2, 0.45]), np.array([0.6, 0.3]))
    array([0.35  , 0.6875])
    """
    if A is None:
        A = np.zeros([len(D)]*2)
    elif order == 'ST':
        A = A.transpose()
    if pc is None:
        pc = np.ones(len(D))/2
    if pr is None:
        pr = 1/(10/pc - 9)

    ### Decomposing interaction matrix as A = A_pos + A_neg, 
        ### where A_pos contains all positive elements of A, while A_neg contains all negatives
    A_pos = np.maximum(A,0)
    A_neg = np.maximum(-A,0)
    d_pos = np.dot(A_pos,D)
    d_neg = np.dot(A_neg,D)

    return (1+d_pos)*pc*(1-D)/(1+d_neg) + (1-pr)*D

def update_equation_quick(D,A,pc,pr):
    """ 
    update_equation updates the probability(ies) to be degraded, D, by one time step, 
    given the interaction matrix A, collapse rate pc and recovery rate pr. 
    Quicker version to go into wrappers and other fns

    Parameters
    ---
    D : numpy.array
        The initial array of degraded probabilities
    A : numpy.array
        The interaction matrix (i.e. adjacency matrix), must be of shape (len(D), len(D)). 
        No self loops, i.e. all diagonal elements should be 0. Must be indexed as A[target,source]
    pc : numpy.array or something broadcastable
        The array/list/collection of collapse rates. Must be in [0,1]. 
    pr : numpy.array or something broadcastable
        The array/list/collection of collapse rates. Must be in [0,1]
        
    Returns
    ---
    D_out : numpy.array

    Examples
    ---
    For an initial vector of zeros, regardless of the interaction matrix, D_out will always be pc
    >>> update_equation(np.zeros(2), np.array( [[0, 4], [-2, 0]] ),np.array([0.2, 0.9]))
    np.array([0.2, 0.9]) 

    For numnerical examples see update_equation
    """
    return (1+np.dot(np.maximum(A,0),D))*pc*(1-D)/(1+np.dot(np.maximum(-A,0),D)) + (1-pr)*D

def temp_response(T,T_lim,beta):
    """ 
    Returns the response in collapse rate to a temperature T, 
    for a tipping element with threshold T_lim and temperature sensitivity (i.e. logistic parameter) beta
    
    Parameters
    ---
    T : numpy.array or float
        Temperature(s)
    T_lim : numpy.array or float
        Tipping/critical threshold(s)
    beta : numpy.array or something broadcastable
        Temperature sensitivity(ies) (i.e. logistic parameter)

    Returns
    ---
    out : numpy.array
    """
    return 1/(1+np.exp(beta*(T_lim-T)))

def evolve(D_init, A, no_time_steps, 
           order_ST=True, 
           pc = None, tau_c = None, 
           T = None, beta = None, T_lim = None,
           pr = None, tau_r = None,
           ):
    """ 
    Wrapper to evolve system up to and including t = no_time_steps, 
    with an initial value of D_init at time t=0. 

    Parameters (required)
    ---
    D_init : numpy.array
        The initial array of degraded probabilities at time step 0
    A : numpy.array
        The interaction matrix (i.e. adjacency matrix), must be of shape (len(D_init), len(D_init)). 
        No self loops, i.e. all diagonal elements should be 0. 
    no_time_steps : int
        The number of time steps to simulate over, 
        i.e. the number of times the update equation is successively applied. 
        Must be >= 1
    
    Parameters (optional):
    order_ST : Bool
        The indexing order of the adjacent matrix. 
        If True, then the adjacency matrix will be read with an index-order that gives A[source, target].
        Otherwise the order will be assumed as A[target, source]
        Default True.
    pc : numpy.array or something broadcastable
        The array/list/collection of collapse rates (without interactions). 
        Will supercede any calls to tau_c or T
        Must be in [0,1]. 
    tau_c :  numpy.array or something broadcastable
        The array/list/collection of collapse time scales. 
    T : array-like or float
        The global temperature(s) at which to evaluate. 
        If array-like must be of length no_time_steps. Must have beta and T_lim supplied as well if not None
    beta : array-like or float
        Temperature sensitivities (i.e. logistic parameters) for individual tipping points.
        If array-like must be of length len(D_init). 
    T_lim : array-like or float
        Critical temperature thresholds for individual tipping points.
        If array-like must be of length len(D_init)
    pr : numpy.array or something broadcastable
        The array/list/collection of collapse rates. Will supercede tau_r
    tau_r : numpy.array or something broadcastable
        The array/list/collection of collapse time scales
    
        
    Returns
    ---
    D_out : numpy.array
        The output will be a (len(D_init), no_time_steps+1)-shaped array where the D_out[:,0] = D_init
    Examples
    ---
    """
    ## Initialising D_out
    D_out = np.zeros((len(D_init), no_time_steps+1))
    D_out[:,0] = D_init
    
    ## Changing pc if needed
    if pc is None:
        if tau_c is None:
            pc = np.ones(len(D_init))/2 ## if nothing is supplied pc will be set to 1/2
        elif (T is None) or (beta is None) or (T_lim is None):
            # if any of T,T_lim,beta are None, then pc = 1/(tau_c+1)
            pc = np.ones(len(D_init))/(tau_c+1)
            print('Non-critical error: at least one of T, T_lim or beta has not been provided.')
        else:
            pc = np.ones((len(D_init), no_time_steps))
            pc /= (tau_c+1)
            for t in no_time_steps:
                pc[:,t] *= temp_response(T[t], T_lim, beta)

    if order_ST:
        A = A.transpose()

    #### STILL NEED TO ACTUALLY DO THE WRAPPER
    return None

In [3]:
update_equation_quick(np.array([0.5, 0.5]), np.array( [[0, 1], [1, 0]] ), np.array([0.2, 0.45]), np.array([0.6, 0.3]))

array([0.35  , 0.6875])

In [21]:
pc = np.ones(9)
pc /= 2
print(pc)

[0.5 0.5 0.5 0.5 0.5 0.5 0.5 0.5 0.5]


# Testing

## Reading individual TE data

In [4]:
folder = '../Data/'
# df_AM22 = pd.read_excel(folder+"Armstrong McKay 2022/science.abn7950_data_s1.xlsx",
#                         skiprows=[0,1,2,3,4],sheet_name='4) Estimates',
#                         skipfooter=40,index_col=1,header=[0,1])
df_TEs = pd.read_excel(folder+'TEsummary.xlsx',header=[0],index_col=0)

df_TEs.head()

,Tipping element,Collapse (years),Recovery (years),Threshold (°C) – Min.,Threshold (°C) – Est.,Threshold (°C) – Max.,Beta (1/°C),Max Impact (°C) – Global,Comments
TE,,,,,,,,,
REEF,Low-latitude coral reefs,10,100,1.0,1.5,2.0,13.813510,0.0,From Armstrong McKay et al no estimate for max...
ASI,Arctic winter sea ice,20,200,4.5,6.3,8.7,3.837086,0.6,NaN
PERM,Boreal permafrost collapse,50,500,3.0,4.0,6.0,6.906755,0.3,From Armstrong McKay 2022 max impact is estima...
AMOC,Atlantic meridional ocean circulation,50,500,1.4,4.0,8.0,2.656444,-0.5,NaN
WAM,Sahel and West African monsoon,50,500,2.0,2.8,3.5,8.633443,0.0,From Armstrong McKay et al no estimate for max...


In [23]:
beta = df_TEs['Beta (1/°C)'].to_numpy()
T_lim = df_TEs['Threshold (°C) – Est.'].to_numpy()
print(temp_response(np.array([0,1]),T_lim,beta))

ValueError: operands could not be broadcast together with shapes (9,) (2,) 

## Reading generated interaction matrices

In [6]:
folder = "../Data/Interaction matrices/"
no_samples = 500
A = [np.loadtxt(folder+str(r)+'.csv') for r in range(no_samples)]

In [15]:
D = np.zeros(9)
T = +1.5
D_out = update_equation(D,A=A[0],pc=temp_response(T,
                                                  df_TEs['Threshold (°C) – Est.'].to_numpy(), 
                                                  df_TEs['Beta (1/°C)']).to_numpy()/ (df_TEs['Collapse (years)'].to_numpy() + 1))

In [16]:
D_out

array([4.54545455e-02, 4.77462617e-10, 6.21606932e-10, 2.55661753e-05,
       2.61896909e-07, 9.91321788e-07, 9.92081741e-09, 2.49875062e-04,
       4.99950005e-05])

In [20]:
help(update_equation)

Help on function update_equation in module __main__:

update_equation(D, A=None, pc=None, pr=None, order='ST')
    update_equation updates the probability(ies) to be degraded, D, by one time step,
    given the interaction matrix A, collapse rate pc and recovery rate pr

    Parameters
    ---
    D : numpy.array
        The initial array of degraded probabilities
    A : numpy.array
        The interaction matrix (i.e. adjacency matrix), must be of shape (len(D), len(D)).
        No self loops, i.e. all diagonal elements should be 0.
        Default is np.zeros([len(D)]*2) i.e. the empty graph, such that all Markov chains are disconnected.
    pc : numpy.array or something broadcastable
        The array/list/collection of collapse rates. Must be in [0,1].
        If none supplied, default is 1/2 for all Markov chains
    pr : numpy.array or something broadcastable
        The array/list/collection of collapse rates. Must be in [0,1]
        If none supplied default is tr = 10tc where